# A/B Test: Listing Accuracy and Value Messaging Checklist

**Purpose:** plan a voluntary host-level randomized pilot and provide a reusable analysis template for prospective results. The existing Airbnb snapshot is observational; it does not contain treatment assignments or post-intervention outcomes, so this notebook does not claim to run an A/B test now.

The tested intervention is a checklist that helps a host verify listing accuracy (photos, amenities, capacity, rules, location description) and clearly explain what is included for the price. The primary outcome is the change in guest-perceived value rating. Rating coverage is tracked alongside it because a listing only has a score when a guest submits a review.


## 1. Design decisions to make before launch

- Recruit hosts who opt in; assign **hosts**, not individual listings, so one host cannot receive both versions and share the checklist across arms.
- Randomize participating hosts within the predeclared host-portfolio segments, using a reproducible seed.
- Treatment hosts receive the checklist for all participating listings. Control hosts continue usual practice during the test, then may be offered the checklist afterward.
- Proposed follow-up: 90 days after launch, with a pre-period of comparable length. Confirm the calendar and review lag before launch.
- Primary outcome: per-host average change in listing-level guest value scores from the pre-period to follow-up. Equal host weighting prevents a few large portfolios from dominating the comparison.
- Guardrail: proportion of randomized hosts/listings that receive at least one value rating in follow-up. Report review counts and rating coverage by arm.
- Set the smallest business-meaningful improvement and the sample size before reviewing outcomes. The effect sizes in the planning cell are scenarios only.


In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import norm, t, ttest_ind
from IPython.display import display

# Find the repository root whether the notebook is launched from the root or scripts/.
_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (p for p in _candidates if (p / "data" / "processed" / "airbnb_features.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate data/processed/airbnb_features.csv")
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "airbnb_features.csv"
PROSPECTIVE_DATA_PATH = PROJECT_ROOT / "data" / "ab_test_listing_outcomes.csv"
RANDOM_SEED = 2026
ALPHA = 0.05
TARGET_POWER = 0.80
ATTRITION_ALLOWANCE = 0.20  # planning assumption; revise before registration

print("Project root:", PROJECT_ROOT)
print("Prospective test file expected at:", PROSPECTIVE_DATA_PATH)


## 2. Use the existing data for planning only

The historical ratings can help estimate rating coverage and host-level variability for rough sample-size planning. They are **not** A/B outcomes, are not a pre/post intervention comparison, and cannot establish a checklist effect. This cell reads the existing processed feature file but does not write to it.


In [ ]:
required = {"host_id", "id", "review_scores_value", "host_portfolio_segment"}
features = pd.read_csv(FEATURES_PATH, usecols=lambda c: c in required)
missing = required.difference(features.columns)
if missing:
    raise ValueError(f"Required planning columns are missing: {sorted(missing)}")

features["review_scores_value"] = pd.to_numeric(features["review_scores_value"], errors="coerce")
rated = features.dropna(subset=["host_id", "review_scores_value"]).copy()

# One historical mean per host: the planning unit matches the proposed randomization unit.
host_baseline = (
    rated.groupby("host_id", as_index=False)
    .agg(
        baseline_value_score=("review_scores_value", "mean"),
        rated_listing_count=("id", "nunique"),
        host_portfolio_segment=("host_portfolio_segment", "first"),
    )
)

print(f"Listings in snapshot: {len(features):,}")
print(f"Listings with value ratings: {len(rated):,} ({len(rated)/len(features):.1%})")
print(f"Hosts with at least one value rating: {len(host_baseline):,}")
print(f"Historical host-mean SD for planning: {host_baseline['baseline_value_score'].std(ddof=1):.4f} points")
print("These are historical planning statistics, not experimental results.")


## 3. Approximate sample-size scenarios

The calculator uses a two-sided, equal-allocation normal approximation for a difference between two independent host-level means. It inflates the result for the declared 20% follow-up attrition assumption. Actual requirements may be larger if review coverage is low or outcomes differ from the historical distribution. Choose the meaningful effect-size threshold with the project stakeholder before starting; do not choose it after seeing test results.


In [ ]:
def approximate_hosts_per_arm(sd: float, detectable_change: float, alpha: float = ALPHA,
                              power: float = TARGET_POWER, attrition: float = ATTRITION_ALLOWANCE) -> int:
    """Normal-approximation planning count for equal-sized arms, inflated for attrition."""
    if sd <= 0 or detectable_change <= 0:
        raise ValueError("sd and detectable_change must be positive")
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)
    evaluable_per_arm = math.ceil(2 * (z_alpha + z_power) ** 2 * sd ** 2 / detectable_change ** 2)
    return math.ceil(evaluable_per_arm / (1 - attrition))

sd_host = host_baseline["baseline_value_score"].std(ddof=1)
scenarios = pd.DataFrame({"illustrative_change_points": [0.10, 0.15, 0.20]})
scenarios["approx_hosts_per_arm_including_attrition"] = scenarios["illustrative_change_points"].map(
    lambda delta: approximate_hosts_per_arm(sd_host, delta)
)
scenarios["approx_total_hosts"] = 2 * scenarios["approx_hosts_per_arm_including_attrition"]
scenarios["assumptions"] = "alpha=0.05, power=0.80, equal allocation, 20% attrition"
scenarios


## 4. Optional host-level randomization (only after opt-in recruitment)

Do not assign the full public snapshot to treatment or control. When a recruitment roster exists, it must contain only consenting, eligible hosts. Keep the assignment roster private; display aggregate balance checks in any shared report.


In [ ]:
def randomize_consented_hosts(participants: pd.DataFrame, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """Assign consenting hosts within portfolio strata; returns IDs for restricted local use only.

    Required columns: host_id, consented, host_portfolio_segment.
    Do not commit or publish the returned row-level assignment roster.
    """
    required = {"host_id", "consented", "host_portfolio_segment"}
    missing = required.difference(participants.columns)
    if missing:
        raise ValueError(f"Participant roster missing columns: {sorted(missing)}")
    consent = participants["consented"]
    if not pd.api.types.is_bool_dtype(consent):
        consent = consent.astype("string").str.strip().str.lower().isin({"true", "1", "yes"})
    roster = participants.loc[consent, ["host_id", "consented", "host_portfolio_segment"]].copy()
    if roster["host_id"].isna().any() or roster["host_id"].duplicated().any():
        raise ValueError("Roster must have one non-missing row per host")
    rng = np.random.default_rng(seed)
    arms = []
    for _, group in roster.groupby("host_portfolio_segment", dropna=False, sort=True):
        indexes = group.index.to_numpy().copy()
        rng.shuffle(indexes)
        treatment_n = len(indexes) // 2
        assigned = {idx: ("treatment" if rank < treatment_n else "control")
                    for rank, idx in enumerate(indexes)}
        arms.extend((idx, assigned[idx]) for idx in group.index)
    arm_map = dict(arms)
    roster["arm"] = roster.index.map(arm_map)
    return roster.reset_index(drop=True)

# Example after approved recruitment (do not fabricate a roster):
# participants = pd.read_csv("restricted/consented_host_roster.csv")
# assignments = randomize_consented_hosts(participants)
# print(assignments.groupby(["host_portfolio_segment", "arm"]).size())


## 5. Prospective data contract

Create `data/ab_test_listing_outcomes.csv` **only after** the pilot. Include every randomized participating listing, even if it receives no review, so coverage can be measured. One row per listing; do not include guest names or raw review text.

| Column | Meaning |
| --- | --- |
| `host_id` | Private cluster/randomization key; do not publish row-level file |
| `listing_id` | Private listing key; do not publish row-level file |
| `arm` | `treatment` or `control`, inherited from the host assignment |
| `baseline_value_score` | Listing mean value score in the predeclared pre-period; null if none |
| `post_value_score` | Listing mean value score in follow-up; null if no new score |
| `post_review_count` | Number of eligible follow-up value scores used for the mean |

Predefine date windows, eligibility, handling of cancelled/withdrawn hosts, and whether any listed fees/amenities are in scope. Use new guest ratings from the same platform scale; do not substitute the old snapshot score for a follow-up outcome.


In [ ]:
def analyze_ab_test(listing_outcomes: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Host-level intention-to-treat summary and Welch comparison of host mean changes.

    Rating change is conditional on a follow-up rating being observed. Coverage is
    therefore reported for all randomized hosts as a required interpretation check.
    """
    required = {"host_id", "listing_id", "arm", "baseline_value_score", "post_value_score", "post_review_count"}
    missing = required.difference(listing_outcomes.columns)
    if missing:
        raise ValueError(f"Outcome file missing columns: {sorted(missing)}")
    d = listing_outcomes.copy()
    if d[["host_id", "listing_id", "arm"]].isna().any().any():
        raise ValueError("host_id, listing_id, and arm must be present for every randomized listing")
    if d["listing_id"].duplicated().any():
        raise ValueError("Outcome file must contain one row per listing")
    if not set(d["arm"].unique()).issubset({"treatment", "control"}):
        raise ValueError("arm values must be 'treatment' or 'control'")
    arms_per_host = d.groupby("host_id")["arm"].nunique(dropna=False)
    if (arms_per_host > 1).any():
        raise ValueError("A host appears in both arms; host-level randomization was violated")
    for col in ["baseline_value_score", "post_value_score"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        invalid = d[col].dropna().between(0, 5).eq(False).any()
        if invalid:
            raise ValueError(f"{col} must be on the 0-5 rating scale")
    d["post_review_count"] = pd.to_numeric(d["post_review_count"], errors="coerce").fillna(0)
    if (d["post_review_count"] < 0).any():
        raise ValueError("post_review_count cannot be negative")
    d["has_post_rating"] = d["post_value_score"].notna()
    by_host = d.groupby(["host_id", "arm"], as_index=False).agg(
        baseline_host_mean=("baseline_value_score", "mean"),
        post_host_mean=("post_value_score", "mean"),
        post_review_count=("post_review_count", "sum"),
        any_followup_rating=("has_post_rating", "any"),
        listing_count=("listing_id", "nunique"),
    )
    by_host["change"] = by_host["post_host_mean"] - by_host["baseline_host_mean"]

    summaries=[]
    for arm, group in by_host.groupby("arm", sort=False):
        valid=group["change"].dropna()
        summaries.append({
            "arm":arm,
            "randomized_hosts":len(group),
            "hosts_with_followup_rating":int(group["any_followup_rating"].sum()),
            "host_rating_coverage":float(group["any_followup_rating"].mean()),
            "hosts_with_baseline_and_followup":len(valid),
            "mean_host_level_change":float(valid.mean()) if len(valid) else np.nan,
            "total_followup_reviews":int(group["post_review_count"].sum()),
        })
    summary=pd.DataFrame(summaries)
    groups={arm:g["change"].dropna().to_numpy() for arm,g in by_host.groupby("arm")}
    effect_row={"contrast":"treatment minus control"}
    if all(a in groups and len(groups[a])>=2 for a in ["treatment","control"]):
        tr,co=groups["treatment"],groups["control"]
        estimate=float(tr.mean()-co.mean())
        v1,v0=tr.var(ddof=1)/len(tr),co.var(ddof=1)/len(co)
        se=math.sqrt(v1+v0)
        df=(v1+v0)**2/(v1**2/(len(tr)-1)+v0**2/(len(co)-1)) if se else np.inf
        crit=t.ppf(0.975,df)
        test=ttest_ind(tr,co,equal_var=False)
        effect_row.update({"mean_change_difference":estimate,"ci95_low":estimate-crit*se,
                           "ci95_high":estimate+crit*se,"welch_p_value":float(test.pvalue),
                           "treatment_hosts_in_score_test":len(tr),"control_hosts_in_score_test":len(co)})
    else:
        effect_row.update({"mean_change_difference":np.nan,"ci95_low":np.nan,"ci95_high":np.nan,
                           "welch_p_value":np.nan,"treatment_hosts_in_score_test":len(groups.get('treatment',[])),
                           "control_hosts_in_score_test":len(groups.get('control',[]))})
    # Descriptive risk difference in follow-up rating coverage; approximate Wald interval.
    cov=summary.set_index('arm')['host_rating_coverage'] if len(summary) else pd.Series(dtype=float)
    if {'treatment','control'}.issubset(cov.index):
        nt=int(summary.set_index('arm').loc['treatment','randomized_hosts'])
        nc=int(summary.set_index('arm').loc['control','randomized_hosts'])
        pt,pc=float(cov['treatment']),float(cov['control'])
        cov_diff=pt-pc
        cov_se=math.sqrt(pt*(1-pt)/nt+pc*(1-pc)/nc) if nt and nc else np.nan
        effect_row.update({"coverage_difference_treatment_minus_control":cov_diff,
                           "coverage_ci95_low":cov_diff-1.96*cov_se,
                           "coverage_ci95_high":cov_diff+1.96*cov_se})
    return summary,pd.DataFrame([effect_row])

if PROSPECTIVE_DATA_PATH.exists():
    prospective=pd.read_csv(PROSPECTIVE_DATA_PATH)
    arm_summary,effect=analyze_ab_test(prospective)
    display(arm_summary)
    display(effect)
else:
    print("No prospective A/B results found. No intervention effect is estimated in this project snapshot.")


## 6. Interpretation and decision rules

- A positive treatment-minus-control change means the treatment arm's observed value score improved more on average, conditional on obtaining follow-up ratings.
- A confidence interval that excludes zero is evidence against no average change under the design and analysis assumptions; it is not proof the checklist works for every host.
- A non-significant result is not proof of no effect. Check whether the planned sample size and follow-up review counts were achieved.
- Compare follow-up rating coverage and review counts by arm. If they differ, the scored listings may not be comparable; explain the potential review-selection bias and avoid presenting the rating difference alone as causal.
- Report the effect estimate and confidence interval, not only the p-value. Keep the host-level randomization intact and do not treat multiple listings from one host as independent assignments.
- If the test shows a useful, reasonably precise improvement without an unacceptable coverage or fairness concern, propose a larger replication. Otherwise revise the checklist or collect more data before deciding.

## Privacy, fairness, and safe use

Recruit by consent. Keep host/listing assignment keys and row-level outcomes in restricted local storage. Do not include names, exact coordinates, guest identifiers, or raw review text in the analysis file. Show aggregate arm-level results and suppress small groups. Offer the checklist to control hosts after the test if feasible. Participation and model scores must not be used for host penalties or listing removal.
